<a href="https://colab.research.google.com/github/perarneskjelvik/Selvstudium-KI/blob/main/DeepFinance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

DeepFinance

Maskinlæring for finans.

In [ ]:
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def data_preprocessing(data, num_lags, train_test_split):
    # Gjør klar dataene for trening
    x = []
    y = []
    for i in range(len(data) - num_lags):
        x.append(data[i:i + num_lags])
        y.append(data[i+ num_lags])
    # Konverter til numpy arrays
    x = np.array(x)
    y = np.array(y)
    # Splitt data i trenings og test set
    split_index = int(train_test_split * len(x))
    x_train = x[:split_index]
    y_train = y[:split_index]
    x_test = x[split_index:]
    y_test = y[split_index:]

    return x_train, y_train, x_test, y_test


def plot_train_test_values(window, train_window, y_train, y_test, y_predicted):
    prediction_window = window
    first = train_window
    second = window - first
    y_predicted = np.reshape(y_predicted, (-1, 1))
    y_test = np.reshape(y_test, (-1, 1))
    plotting_time_series = np.zeros((prediction_window, 3))
    plotting_time_series[0:first, 0] = y_train[-first:].flatten()
    plotting_time_series[first:, 1] = y_test[0:second, 0]
    plotting_time_series[first:, 2] = y_predicted[0:second, 0]
    plotting_time_series[0:first, 1] = np.nan
    plotting_time_series[0:first, 2] = np.nan
    plotting_time_series[first:, 0] = np.nan
    plt.plot(plotting_time_series[:, 0], label = 'Trenings data', color = 'black', linewidth = 2.5)
    plt.plot(plotting_time_series[:, 1], label = 'Test data', color = 'black', linestyle = 'dashed', linewidth = 2)
    plt.plot(plotting_time_series[:, 2], label = 'Prognose data', color = 'red', linewidth = 1)
    plt.axvline(x = first, color = 'black', linestyle = '--', linewidth = 1)
    plt.grid()
    plt.legend()


def calculate_accuracy(predicted_returns, real_returns):
    predicted_returns = np.reshape(predicted_returns, (-1, 1))
    real_returns = np.reshape(real_returns, (-1, 1))
    hits = sum((np.sign(predicted_returns)) == np.sign(real_returns))
    total_samples = len(predicted_returns)
    accuracy = hits / total_samples

    return accuracy[0] * 100

def model_bias(predicted_returns):
    bullish_forecasts = np.sum(predicted_returns > 0)
    bearish_forecasts = np.sum(predicted_returns < 0)

    # Unngp å dele på null
    if bearish_forecasts == 0:
        if bullish_forecasts > 0:
            return np.inf  # Alle prognoser er positive
        else:
            return 0.0   # Ingen prognoser

    return bullish_forecasts / bearish_forecasts


# Hovedprogram

# Maskinlæring for å lage prognose på daglig Equinor avkastning.
# Bruker Yahoo Finance for å hente historiske kurser.


from keras.models import Sequential
from keras.layers import Dense, Input, Dropout # Import Dropout lag
from keras.callbacks import EarlyStopping # Import EarlyStopping
import keras
import numpy as np
import matplotlib.pyplot as plt
import pandas_datareader as pdr
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import RobustScaler # Import RobustScaler

# Installerer yfinance for Yahoo Finance data
!pip install yfinance
import yfinance as yf

# Setter start og slutt dato for dataene
start_date = '2018-06-18' # start dato for EQNR (NYSE listing)
end_date   = '2026-06-01'

# Henter aksjekurser fra Yahoo Finance
df = yf.download('EQNR', start=start_date, end=end_date)
data = df['Close'].dropna().to_numpy().flatten() # Ensure data is a 1D numpy array


# Beregner forskjellen mellom dagens kurs og gårsdagens kurs
data = np.diff(data)


# Setter hyperparametere
num_lags = 50
train_test_split = 0.80
num_neurons_in_hidden_layers = 20
num_epochs = 500
batch_size = 32


# Lager trenings og test sett
x_train, y_train, x_test, y_test = data_preprocessing(data, num_lags, train_test_split)

# Standardiserer inndata

scaler_x = RobustScaler()
x_train_scaled = scaler_x.fit_transform(x_train)
x_test_scaled = scaler_x.transform(x_test)

scaler_y = RobustScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1))

x_train = x_train_scaled
x_test = x_test_scaled
y_train = y_train_scaled
y_test = y_test_scaled


# Lager modellen
model = Sequential()

model.add(Input(shape=(num_lags,))) # Bruker Input layer eksplisitt
model.add(Dense(num_neurons_in_hidden_layers, activation = 'relu'))
model.add(Dropout(0.2)) # Bruker Dropout layer med dropout rate lik 0.2

model.add(Dense(num_neurons_in_hidden_layers, activation = 'relu'))

# Output layer
model.add(Dense(1))

# Kompilerer
model.compile(loss = 'mean_squared_error', optimizer = 'adam')

# Definerer EarlyStopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

# Trener modellen
model.fit(x_train, np.reshape(y_train, (-1, 1)), epochs = num_epochs, batch_size = batch_size,
          validation_split=0.2, callbacks=[early_stopping])

# Predicting in-sample (scaled predictions)
y_predicted_train_scaled = np.reshape(model.predict(x_train), (-1, 1))

# Predicting out-of-sample (scaled predictions)
y_predicted_scaled = np.reshape(model.predict(x_test), (-1, 1))

# Omformer prognoser og faktiske verdier tilbake til origanal skal for evaluering
y_predicted_train = scaler_y.inverse_transform(y_predicted_train_scaled)
y_predicted = scaler_y.inverse_transform(y_predicted_scaled)
y_train_unscaled = scaler_y.inverse_transform(y_train)
y_test_unscaled = scaler_y.inverse_transform(y_test)

# Plotter
plot_train_test_values(100, 50, y_train_unscaled, y_test_unscaled, y_predicted)

# Evaluering av modellen
print('---')
print('--- Equinoe Daglig')
print('--- Evaluering')
print('Nøyaktighet Trening = ', round(calculate_accuracy(y_predicted_train, y_train_unscaled), 2), '%')
print('Nøyaktighet Test = ', round(calculate_accuracy(y_predicted, y_test_unscaled), 2), '%')
print('RMSE Train = ', round(np.sqrt(mean_squared_error(y_predicted_train, y_train_unscaled)), 10))
print('Standardavvik (RMSE) Test = ', round(np.sqrt(mean_squared_error(y_predicted, y_test_unscaled)), 10))
print('Correlation In-Sample Predicted/Train = ', round(np.corrcoef(np.reshape(y_predicted_train, (-1)), y_train_unscaled.flatten())[0][1], 3))
print('Correlation Out-of-Sample Predicted/Test = ', round(np.corrcoef(np.reshape(y_predicted, (-1)), np.reshape(y_test_unscaled, (-1)))[0][1], 3))
print('Modell skjevhet (opp/ned) (Bias) = ', round(model_bias(y_predicted), 2))
print('---')
